In [40]:
from qiskit.transpiler import CouplingMap
from qiskit_addon_utils.problem_generators import generate_xyz_hamiltonian

num_spins = 22
coupling_map = CouplingMap.from_ring(num_spins)
H_op = generate_xyz_hamiltonian(coupling_map, coupling_constants=(0.3, 0.3, 1.0))

In [43]:
# Set parameters for quantum Krylov algorithm
krylov_dim = 5  # size of krylov subspace
dt = 0.15
num_trotter_steps = 6

In [44]:
# Prep `Neel` state as the reference state for evolution
from qiskit import QuantumCircuit

qc_state_prep = QuantumCircuit(num_spins)
for i in range(num_spins):
    if i % 2 == 0:
        qc_state_prep.x(i)

In [45]:
from qiskit.circuit import QuantumRegister
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.synthesis import LieTrotter

evol_gate = PauliEvolutionGate(
    H_op, time=(dt / num_trotter_steps), synthesis=LieTrotter(reps=num_trotter_steps)
)  # `U` operator

qr = QuantumRegister(num_spins)
qc_evol = QuantumCircuit(qr)
qc_evol.append(evol_gate, qargs=qr)

circuits = []
for rep in range(krylov_dim):
    circ = qc_state_prep.copy()

    # Repeating the `U` operator to implement U^0, U^1, U^2, and so on, for power Krylov space
    for _ in range(rep):
        circ.compose(other=qc_evol, inplace=True)

    circ.measure_all()
    circuits.append(circ)

In [108]:
from qiskit import transpile
from qiskit_aer import AerSimulator

sim = AerSimulator()

# translate the circuit(s) to instructions Aer understands
sim_circuits = transpile(circuits, backend=sim)

job = sim.run(sim_circuits, shots=1000)
result = job.result()

In [109]:
counts_all = [result.get_counts(i) for i in range(krylov_dim)]

In [110]:
from collections import Counter

counts_cumulative = []
for i in range(krylov_dim):
    counter = Counter()
    for d in counts_all[: i + 1]:
        counter.update(d)

    counts = dict(counter)
    counts_cumulative.append(counts)

In [111]:
print(counts_cumulative)

[{'0101010101010101010101': 1000}, {'0101010101010101010101': 1999, '0101010101010011010101': 1}, {'0101010101010101010101': 2986, '0101010101010011010101': 2, '0101010101010101011001': 1, '0101010101010100110101': 2, '0101010101010101010011': 1, '0100110101010101010101': 2, '0101010101010101010110': 1, '0101010101011001010101': 2, '1001010101010101010101': 1, '1001010110010101010101': 1, '0101011001010101010101': 1}, {'0101010101010101010101': 3939, '0101010101010011010101': 6, '0101010101010101011001': 3, '0101010101010100110101': 5, '0101010101010101010011': 1, '0100110101010101010101': 2, '0101010101010101010110': 4, '0101010101011001010101': 4, '1001010101010101010101': 4, '1001010110010101010101': 1, '0101011001010101010101': 2, '0011010101010101010101': 4, '0101010101010101100101': 3, '0101100101010101010101': 6, '0101010011010101010101': 2, '0101011010010101010101': 1, '0110010101010101010101': 1, '0101010101010110010101': 2, '0101001101010101010101': 2, '0101010101010101001101

In [112]:
bitstrings_skqd = {k for d in counts_all for k in d}

In [113]:
from schwingermodel import project_hamiltonian_to_bitstring_subspace, diagonalize_projected_hamiltonian, exact_ground_state_energy

In [114]:
H_sub, basis = project_hamiltonian_to_bitstring_subspace(H_op, bitstrings_skqd)

evals, evecs = diagonalize_projected_hamiltonian(H_sub)
print('vs. approximated via Krylov: ', min(evals))

vs. approximated via Krylov:  -23.46107255910789


In [115]:
print(len(bitstrings_skqd))

28


In [42]:
assert False

AssertionError: 

In [64]:
from schwingermodel import bitflip_projection, reachable_bitstrings_accumulating

In [67]:
def get_initial_bitstring(qc):
    n = qc.num_qubits
    bits = ['0'] * n

    for inst in qc.data:
        if inst.operation.name == "x":
            q = inst.qubits[0]._index
            bits[q] = '1'

    return ''.join(reversed(bits))  # reverse for usual bitstring order
initial_state = get_initial_bitstring(qc_state_prep)

# Bark

In [116]:
from BARK import BARK

In [117]:
BK = BARK(H_op, initial_state, max_iterations=2, time_step=1, tolerance=0.0)

bases = BK.basis

                                                          █                                        
                                                          ██                                       
                                                         █ █                     █                 
                                                         █  █                 ██ █                 
                                                        █   █               ██   █                 
                                                      ██     █            ██     █                 
                                                     ███     █          ██       █                 
                                                   ██        █        ██         █                 
                                                ██           █     ██  █        ██                 
                                              ██             █  ███   █         ██                 


In [120]:
bitstrings = bases[2]
print(len(bitstrings))

254


In [121]:
import random
rand = False
N = abs(len(bitstrings) - len(bitstrings_skqd))
if not rand: 
    bitstrings = bitstrings[:-N]
else: 
    for i in sorted(random.sample(range(len(bitstrings)), N), reverse=True):
        bitstrings.pop(i)

In [123]:
H_sub, basis = project_hamiltonian_to_bitstring_subspace(H_op, bitstrings)

evals, evecs = diagonalize_projected_hamiltonian(H_sub)
print('vs. approximated via Krylov: ', min(evals))
print(f'number of bitstrings: {len(bitstrings)}')

vs. approximated via Krylov:  -22.51371806340166
number of bitstrings: 28
